In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, RobustScaler
from sklearn.metrics import silhouette_score, make_scorer
import joblib
import seaborn as sns
from sklearn.model_selection import GridSearchCV, KFold
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pygeohash as pgh
import geopy as gpy
import numpy as np
import warnings
from sklearn.decomposition import PCA
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.mixture import GaussianMixture
import os
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

%load_ext autoreload
%autoreload 2

In [ ]:
filename = 'research_centers.csv'
path_to_data = f'data/{filename}'
columns_to_drop = ['researchCenterId', 'researchCenterName']
geographical_cols = ['latitude', 'longitude']
count_data_cols = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km']
ratio_cols = ['facilityDiversity_10km', 'facilityDensity_10km']
categorical_cols = ['city']
colors = plt.cm.tab20.colors
geohash_precision = 4
geolocater = gpy.geocoders.Nominatim(user_agent="research_grid_geocoder")
os.makedirs('models', exist_ok=True)

# Load

In [ ]:
raw_df = pd.read_csv(path_to_data)
raw_df.head()

In [ ]:
raw_df.info(), raw_df.duplicated().sum()

# Explore

In [ ]:
raw_df['researchCenterName'].nunique(), raw_df['city'].nunique(), raw_df.columns

In [ ]:
df = raw_df.set_index('researchCenterId')
for geohash_num in range(2, 10):
    df[f'geohash_{geohash_num}'] = df.apply(lambda x: pgh.encode(x['latitude'], x['longitude'], precision = geohash_num), axis=1)
df['external_score'] = df['hospitals_10km'] + df['pharmacies_10km']
df['internal_external_ratio'] = df['internalFacilitiesCount'] / (df['hospitals_10km'] + df['pharmacies_10km'] + 1) 
df['healthcare_deficit'] = (df['hospitals_10km'] + df['pharmacies_10km']) == 0
df['lat_rad'] = np.radians(df['latitude'])
df['lon_rad'] = np.radians(df['longitude'])
df['cart_x'] = np.cos(df['lat_rad']) * np.cos(df['lon_rad'])
df['cart_y'] = np.cos(df['lat_rad']) * np.sin(df['lon_rad'])
df['cart_z'] = np.sin(df['lat_rad'])
df.drop(columns =['lat_rad', 'lon_rad'], inplace=True)
# coordinates_series = df['latitude'].astype(str).str.cat(df['longitude'].astype(str), sep=',') 
df.head()

In [ ]:
df[['city','geohash_2', 'geohash_3', 'geohash_4']].nunique()

In [ ]:
df.groupby('city').agg(**{
    'precision 2 unique': ('geohash_2', 'nunique'),
    'precision 2 names': ('geohash_2', 'unique'),
    'precision 3 unique': ('geohash_3', 'nunique'),
    'precision 3 names': ('geohash_3', 'unique'),
    'precision 4 unique': ('geohash_4', 'nunique'),
    'precision 4 names': ('geohash_4', 'unique')
})

### Research Centers - each record is a research center

In [ ]:
df[['researchCenterName']].nunique()

Filtering needs to be done by examinining and looking at the city heat map

### City

In [ ]:
df['city'].nunique(), df['city'].unique()

In [ ]:
city_summary_df = df.groupby('city').agg(**{
    'Total Research Centers': ('researchCenterName', 'nunique'),
    'Total Facilities': ('internalFacilitiesCount', 'sum'),
    'Total Hospitals': ('hospitals_10km', 'sum'),
    'Total Pharmacies': ('pharmacies_10km', 'sum')
})
city_summary_df.loc['Total', :]  = city_summary_df.sum()
city_summary_df.loc[:,'Research Distribution'] = round(city_summary_df['Total Research Centers'] / city_summary_df.loc['Total', 'Total Research Centers'] * 100, 2)
city_summary_df.loc[:,'Facility Distribution'] = round(city_summary_df['Total Facilities'] / city_summary_df.loc['Total', 'Total Facilities'] * 100, 2)
city_summary_df.loc[:,'Hospital Distribution'] = round(city_summary_df['Total Hospitals'] / city_summary_df.loc['Total', 'Total Hospitals'] * 100, 2)
city_summary_df.loc[:,'Pharmacy Distribution'] = round(city_summary_df['Total Pharmacies'] / city_summary_df.loc['Total', 'Total Pharmacies'] * 100, 2)

city_summary_df

In [ ]:
geohash_summary_list = []

for a_geohash in ['geohash_2', 'geohash_3', 'geohash_4']:
    temp_geohash_summary_df = df.groupby(a_geohash).agg(**{
        'Total Research Centers': ('researchCenterName', 'nunique'),
        'Total Facilities': ('internalFacilitiesCount', 'sum'),
        'Total Hospitals': ('hospitals_10km', 'sum'),
        'Total Pharmacies': ('pharmacies_10km', 'sum')
    })
    temp_geohash_summary_df.loc[f'Total {a_geohash}', :]  = temp_geohash_summary_df.sum()
    temp_geohash_summary_df.loc[:,'Research Distribution'] = round(temp_geohash_summary_df['Total Research Centers'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Research Centers'] * 100, 2)
    temp_geohash_summary_df.loc[:,'Facility Distribution'] = round(temp_geohash_summary_df['Total Facilities'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Facilities'] * 100, 2)
    temp_geohash_summary_df.loc[:,'Hospital Distribution'] = round(temp_geohash_summary_df['Total Hospitals'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Hospitals'] * 100, 2)
    temp_geohash_summary_df.loc[:,'Pharmacy Distribution'] = round(temp_geohash_summary_df['Total Pharmacies'] / temp_geohash_summary_df.loc[f'Total {a_geohash}', 'Total Pharmacies'] * 100, 2)

    temp_geohash_summary_df.drop(index=f'Total {a_geohash}', inplace=True)
    temp_geohash_summary_df = pd.concat([temp_geohash_summary_df.rename_axis('city')], axis=0, keys=[a_geohash])

    geohash_summary_list.append(temp_geohash_summary_df)

geohash_summary_df = pd.concat(geohash_summary_list, axis=0)
geohash_summary_df.head()

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="city",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="geohash_2",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="geohash_3",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

In [ ]:
geohash_summary_df.loc['geohash_3']

In [ ]:
fig = px.scatter(
    df,
    x="latitude",
    y="longitude",
    color="geohash_4",
    hover_data=["researchCenterName"]
)

fig.update_layout(title="City Locations by Coordinates")
fig.show()

### Internal Facilities Count

In [ ]:
facility_summary_df = df.groupby('city').agg(**{
    'Average Facilities': ('internalFacilitiesCount', 'mean'),
    'Total Facilities': ('internalFacilitiesCount', 'sum'),
    'Number of Research Centers': ('researchCenterName', 'nunique')
})
facility_summary_df

In [ ]:
geohash_facility_summary_df = df.groupby('geohash_3').agg(**{
    'Average Facilities': ('internalFacilitiesCount', 'mean'),
    'Total Facilities': ('internalFacilitiesCount', 'sum'),
    'Number of Research Centers': ('researchCenterName', 'nunique')
})
geohash_facility_summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
facility_summary_df.plot(kind='bar', 
                         y='Total Facilities', 
                         legend=False, 
                         color=colors[:len(facility_summary_df)],
                         ax=ax)
ax.set_title('Total Facilities by City')
ax.set_xlabel('City')
ax.tick_params(axis='x', rotation=0)
ax.set_ylabel('Total Facilities')
ax.grid(axis='y', linestyle='--', alpha=0.7)

In [ ]:
df['internalFacilitiesCount'].hist(bins=11, color='skyblue', edgecolor='black')
plt.title('Distribution of Internal Facilities Count')
plt.xlabel('Internal Facilities Count')
plt.ylabel('Frequency')

In [ ]:
df['internalFacilitiesCount'].plot(kind='box', vert=False, color='skyblue')
plt.title('Box Plot of Internal Facilities Count')
plt.xlabel('All Research Centers')

In [ ]:
cities = df['city'].sort_values().unique()
max_bin_count = df['internalFacilitiesCount'].max()

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

for i, city in enumerate(cities):
    subset = df.loc[df['city'] == city, 'internalFacilitiesCount']
    subset.plot(
        kind='hist',
        bins=max_bin_count,
        ax=axes[i],
        edgecolor='black',
        alpha=0.7,
        title=city,
        sharex=False,
        sharey=True
    )
    axes[i].set_xlabel('Facility Count')
    axes[i].set_ylabel('Frequency')
    axes[i].set_xlim(0, max_bin_count + 1)
    axes[i].set_ylim(0, 4)

for j in range(len(cities), len(axes)):
    axes[j].axis('off')

fig.suptitle('internalFacilitiesCount Distribution by City (3x2 grid)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
cities = df['geohash_3'].sort_values().unique()
max_bin_count = df['internalFacilitiesCount'].max()

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

for i, city in enumerate(cities):
    subset = df.loc[df['geohash_3'] == city, 'internalFacilitiesCount']
    subset.plot(
        kind='hist',
        bins=max_bin_count,
        ax=axes[i],
        edgecolor='black',
        alpha=0.7,
        title=city,
        sharex=False,
        sharey=True
    )
    axes[i].set_xlabel('Facility Count')
    axes[i].set_ylabel('Frequency')
    axes[i].set_xlim(0, max_bin_count + 1)
    axes[i].set_ylim(0, 4)

for j in range(len(cities), len(axes)):
    axes[j].axis('off')

fig.suptitle('internalFacilitiesCount Distribution by City (3x2 grid)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
df.boxplot(column='internalFacilitiesCount', by='city', vert=False, figsize=(10, 6))
plt.title('Box Plot of Internal Facilities Count by City')
plt.suptitle('')

In [ ]:
df.boxplot(column='internalFacilitiesCount', by='geohash_3', vert=False, figsize=(10, 6))
plt.title('Box Plot of Internal Facilities Count by Geohash')
plt.suptitle('')

### Hospital and pharmacies

In [ ]:
df.groupby('city').agg(**{
    'Total Hospitals': ('hospitals_10km', 'sum'),
    'Total Pharmacies': ('pharmacies_10km', 'sum')
})

In [ ]:
df.plot.box(column=['hospitals_10km', 'pharmacies_10km'], by='city', vert=False, figsize=(16, 8))

In [ ]:
df.plot.box(column=['hospitals_10km', 'pharmacies_10km'], by='geohash_3', vert=False, figsize=(16, 8))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), ncols=2, nrows=1)
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='internalFacilitiesCount', 
    hue='city',          
    palette='rocket', 
    alpha=0.75,
    ax=ax[0]
)
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='internalFacilitiesCount', 
    hue='geohash_3',          
    palette='rocket', 
    alpha=0.75,
    ax=ax[1]
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='pharmacies_10km', 
    hue='city',          
    palette='rocket', 
    alpha=0.75
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='facilityDiversity_10km', 
    hue='city',          
    palette='viridis', 
    alpha=0.75
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='hospitals_10km', 
    y='facilityDensity_10km', 
    hue='city',          
    palette='rocket', 
    alpha=0.75
)
plt.show()

### Facility Diversity

In [ ]:
df['facilityDiversity_10km'].plot(kind='box', vert=False, color='skyblue')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), ncols=2, nrows=1)
df.plot(kind='box', column='facilityDiversity_10km', by='city', vert=False, figsize=(14, 12), subplots=True, ax=ax[0])
df.plot(kind='box', column='facilityDiversity_10km', by='geohash_3', vert=False, figsize=(14, 12), subplots=True, ax=ax[1])

### Facility Density

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), ncols=2, nrows=1)
df.plot(kind='box', column='facilityDensity_10km', by='city', vert=False, figsize=(14, 12), ax=ax[0])
df.plot(kind='box', column='facilityDensity_10km', by='geohash_3', vert=False, figsize=(14, 12), ax=ax[1])

### Correlation

In [ ]:
df.head()

In [ ]:
numeric_cols = ['internalFacilitiesCount','hospitals_10km','pharmacies_10km','facilityDiversity_10km','facilityDensity_10km', 'external_score', 'internal_external_ratio','cart_x', 'cart_y', 'cart_z', 'healthcare_deficit']
correlation = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation heatmap of key numeric features')
plt.show()

corr_with_diversity = correlation['facilityDiversity_10km'].sort_values(ascending=False)
corr_with_internal = correlation['internalFacilitiesCount'].sort_values(ascending=False)
print('Correlation with facilityDiversity_10km:\n', corr_with_diversity)
print('\nCorrelation with internalFacilitiesCount:\n', corr_with_internal)

# Feature Selection, Preprocessing, and K-Means Clustering

We will use scikit-learn transformers to build a pipeline that standardizes relevant numeric features and fits K-Means. This is purely unsupervised for now (no Gold labels), then we map clusters to `Premium`, `Standard`, `Basic` by center quality signal.

### Baseline Model

In [ ]:
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('geographical', StandardScaler(), ['latitude', 'longitude']),
        ('categorical', OneHotEncoder(), ['city']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km'])
    ],
    remainder='drop'
)

X_reference = baseline_preprocessor.fit_transform(df)

results = []
random_labels = np.random.randint(0, 3, size=len(df))
score = silhouette_score(X_reference, random_labels)
results.append({'experiment': 'Random Assignment', 'silhouette': score})

gmm = GaussianMixture(n_components=3, random_state=42)
labels = gmm.fit_predict(X_reference)
score = silhouette_score(X_reference, labels)
results.append({'experiment': 'GMM', 'silhouette': score})

baseline_df = pd.DataFrame(results)
baseline_df

### Phase 2

In [ ]:
latitude_longitude_preprocessor = ColumnTransformer(
    transformers=[
        ('geographical', StandardScaler(), ['latitude', 'longitude']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio'])
    ],
    remainder='drop'
)

cart_preprocessor = ColumnTransformer(
    transformers=[
        ('numerical_geo', StandardScaler(), ['cart_x', 'cart_y', 'cart_z']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio'])
    ],
    remainder='drop'
)

geocache_preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(), ['geohash_3']),
        ('numerical', StandardScaler(), ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio'])
    ],
        remainder='drop'
)

experiments = {
    'naive': baseline_preprocessor,
    'coords_only': latitude_longitude_preprocessor,
    'cartesian': cart_preprocessor,
    'geohash': geocache_preprocessor
}

results = []
for name, transformer in experiments.items():
    X_processed = transformer.fit_transform(df)

    for k in range(2, 10):
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_processed)
        score = silhouette_score(X_processed, labels)
        
        results.append({'experiment': name, 'k': k, 'silhouette': score, 'inertia': model.inertia_})

phase_2_exp = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(10, 6), ncols = 2, nrows=1)
exp_pivot_df = phase_2_exp.pivot(index='k', columns='experiment', values='silhouette')
exp_pivot_inertia_df = phase_2_exp.pivot(index='k', columns='experiment', values='inertia')
exp_pivot_df.plot(marker='o', ax=ax[0])
exp_pivot_inertia_df.plot(marker='o', ax=ax[1])
ax[0].set_title('Silhouette Scores by Experiment')
ax[0].set_xlabel('Number of Clusters (k)')
ax[0].set_ylabel('Silhouette Score')
ax[1].set_title('Inertia by Experiment')
ax[1].set_xlabel('Number of Clusters (k)')
ax[1].set_ylabel('Inertia')
plt.tight_layout()

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
geo_cols = ['latitude', 'longitude', 'city', 'geohash_2', 'geohash_3', 'geohash_4', 'geohash_5', 'geohash_6', 'geohash_7', 'geohash_8', 'geohash_9', 'cart_x', 'cart_y', 'cart_z']
num_cols = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km', 'facilityDiversity_10km', 'facilityDensity_10km', 'external_score', 'internal_external_ratio']

experiments = {
    'baseline': baseline_preprocessor,
    'simple_geohash': geocache_preprocessor,
}

for geocache_num in range(2, 10):
    for pca_num in range(2, 10):
        target_col = f'geohash_{geocache_num}'
        to_drop = ['researchCenterName'] + [c for c in geo_cols if c != target_col]
        
        base_transformer = ColumnTransformer([
            ('drop', 'drop', to_drop),
            ('geo_cat', OneHotEncoder(), [target_col]),
            ('numerical', StandardScaler(), num_cols)
        ])
        
        experiments[f'geohash_{geocache_num}_pca_{pca_num}'] = Pipeline([
            ('prep', base_transformer),
            ('pca', PCA(n_components=pca_num, random_state=42))
        ])

results = []
for name, transformer in experiments.items():
    X_processed = transformer.fit_transform(df)

    for k in range(2, 10):
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_processed)
        score = silhouette_score(X_processed, labels)
        
        results.append({'experiment': name, 'k': k, 'silhouette': score, 'inertia': model.inertia_})

phase_3_exp = pd.DataFrame(results)
phase_3_exp[phase_3_exp['k'] == 3].sort_values(by='silhouette', ascending=False).head()


In [ ]:
master_results_df = pd.concat([phase_3_exp.sort_values(by='silhouette', ascending=False).head(1), 
           phase_3_exp[phase_3_exp['k'] == 3].sort_values(by='silhouette', ascending=False).head(1),
           baseline_df]).fillna('N/A').reset_index(drop=True)
master_results_df

In [ ]:
exp_pivot_df = phase_3_exp.pivot(index='k', columns='experiment', values='silhouette')
exp_pivot_inertia_df = phase_3_exp.pivot(index='k', columns='experiment', values='inertia')

experiments_list = list(exp_pivot_df.columns)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Silhouette Scores by Experiment', 'Inertia by Experiment'),
    horizontal_spacing=0.12
)

for experiment in experiments_list:
    fig.add_trace(
        go.Scatter(
            x=exp_pivot_df.index,
            y=exp_pivot_df[experiment],
            mode='lines+markers',
            name=experiment,
            legendgroup=experiment,
            visible=True,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate=f'<b>{experiment}</b><br>k=%{{x}}<br>Silhouette=%{{y:.3f}}<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=exp_pivot_inertia_df.index,
            y=exp_pivot_inertia_df[experiment],
            mode='lines+markers',
            name=experiment,
            legendgroup=experiment,
            showlegend=False,
            visible=True,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate=f'<b>{experiment}</b><br>k=%{{x}}<br>Inertia=%{{y:.0f}}<extra></extra>'
        ),
        row=1, col=2
    )

fig.update_layout(
    title_text="Clustering Evaluation Metrics",
    height=500,
    width=1200,
    hovermode='closest',
    legend=dict(
        title="Experiments",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="black",
        borderwidth=1
    )
)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=1)
fig.update_yaxes(title_text="Inertia", row=1, col=2)

dropdown_buttons = []
for experiment in experiments_list:
    visibility = []
    for exp in experiments_list:
        visibility.append(exp == experiment)
    visibility = visibility + visibility
    
    dropdown_buttons.append(
        dict(
            label=experiment,
            method="update",
            args=[{"visible": visibility}]
        )
    )

# Add "Show All" button
all_visible = [True] * (len(experiments_list) * 2)
dropdown_buttons.insert(0, dict(
    label="Show All",
    method="update",
    args=[{"visible": all_visible}]
))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            showactive=True,
            x=0.02,
            y=1.15,
            xanchor="left",
            yanchor="top",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(size=12)
        )
    ]
)

fig.show()

# Analysis

In [ ]:
# =========================
# FINAL MODEL (KMEANS MVP)
# =========================
mvp_exp = phase_3_exp[phase_3_exp['k'] == 3] \
    .sort_values(by='silhouette', ascending=False) \
    .reset_index(drop=True).loc[0, 'experiment']

final_pipeline = Pipeline([
    ('preprocessor', experiments[mvp_exp]),
    ('kmeans', KMeans(n_clusters=3, random_state=42, n_init=10))
])

final_pipeline.fit(df)

df_kmeans = df.copy()
df_kmeans['cluster'] = final_pipeline.named_steps['kmeans'].labels_

# Quality score
def compute_quality(df_):
    return (
        df_['internalFacilitiesCount'] * 0.3 +
        df_['hospitals_10km'] * 0.2 +
        df_['pharmacies_10km'] * 0.15 +
        df_['facilityDiversity_10km'] * 0.2 +
        df_['facilityDensity_10km'] * 0.15
    )

df_kmeans['quality_score'] = compute_quality(df_kmeans)

# Map clusters → tiers
tiers = ['Basic', 'Standard', 'Premium']
cluster_quality = df_kmeans.groupby('cluster')['quality_score'].mean().sort_values()

mapping_kmeans = {}
for i, (cluster, _) in enumerate(cluster_quality.items()):
    mapping_kmeans[cluster] = tiers[i]

df_kmeans['qualityTier'] = df_kmeans['cluster'].map(mapping_kmeans)
df_kmeans['model'] = 'KMeans (MVP)'


# =========================
# BASELINE (GMM)
# =========================
baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('geo', StandardScaler(), ['latitude', 'longitude']),
        ('cat', OneHotEncoder(), ['city']),
        ('num', StandardScaler(), [
            'internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km',
            'facilityDiversity_10km', 'facilityDensity_10km'
        ])
    ]
)

X_ref = baseline_preprocessor.fit_transform(df)

df_gmm = df.copy()
gmm = GaussianMixture(n_components=3, random_state=42)
df_gmm['cluster'] = gmm.fit_predict(X_ref)

df_gmm['quality_score'] = compute_quality(df_gmm)

cluster_quality_gmm = df_gmm.groupby('cluster')['quality_score'].mean().sort_values()

mapping_gmm = {}
for i, (cluster, _) in enumerate(cluster_quality_gmm.items()):
    mapping_gmm[cluster] = tiers[i]

df_gmm['qualityTier'] = df_gmm['cluster'].map(mapping_gmm)
df_gmm['model'] = 'Baseline (GMM)'


# =========================
# RANDOM BASELINE
# =========================
df_random = df.copy()

np.random.seed(42)
df_random['cluster'] = np.random.randint(0, 3, size=len(df_random))

df_random['quality_score'] = compute_quality(df_random)

cluster_quality_rand = df_random.groupby('cluster')['quality_score'].mean().sort_values()

mapping_rand = {}
for i, (cluster, _) in enumerate(cluster_quality_rand.items()):
    mapping_rand[cluster] = tiers[i]

df_random['qualityTier'] = df_random['cluster'].map(mapping_rand)
df_random['model'] = 'Random'


# =========================
# COMBINE ALL
# =========================
combined_df = pd.concat([df_kmeans, df_gmm, df_random], ignore_index=True)


# =========================
# BOXPLOTS (2 COLUMNS GRID)
# =========================
features = [
    'internalFacilitiesCount',
    'hospitals_10km',
    'pharmacies_10km',
    'facilityDiversity_10km',
    'facilityDensity_10km'
]

n_plots = len(features) + 1  # +1 for count plot
n_cols = 2
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.boxplot(
        data=combined_df,
        x='qualityTier',
        y=feature,
        hue='model',
        order=['Premium', 'Standard', 'Basic'],
        ax=axes[i]
    )
    axes[i].set_title(f'{feature} by Tier')

# Count plot
tier_counts = combined_df.groupby(['qualityTier', 'model']).size().unstack()
tier_counts.plot(kind='bar', ax=axes[len(features)])
axes[len(features)].set_title('Number of Centers per Tier')
axes[len(features)].set_ylabel('Count')

# Clean legends
for ax in axes:
    if ax.get_legend():
        ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')

plt.tight_layout()
plt.show()


# =========================
# SCATTER (ONLY MVP)
# =========================
plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    df_kmeans['internalFacilitiesCount'],
    df_kmeans['hospitals_10km'],
    c=df_kmeans['qualityTier'].map({'Premium': 2, 'Standard': 1, 'Basic': 0}),
    cmap='RdYlGn',
    alpha=0.7,
    s=100
)

plt.xlabel('Internal Facilities Count')
plt.ylabel('Hospitals within 10km')
plt.title('MVP (KMeans): Internal Facilities vs Hospitals')

plt.colorbar(scatter, label='Tier (2=Premium, 1=Standard, 0=Basic)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# K=3 model (from df_kmeans)
scatter1 = axes[0].scatter(
    df_kmeans['internalFacilitiesCount'], 
    df_kmeans['hospitals_10km'], 
    c=df_kmeans['cluster'], 
    cmap='viridis', 
    alpha=0.7,
    s=80
)
axes[0].set_title('K-Means (k=3) - Silhouette Score: 0.61')
axes[0].set_xlabel('Internal Facilities Count')
axes[0].set_ylabel('Hospitals within 10km')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# K=5 model (re-run with k=5)
X_processed_5 = experiments[mvp_exp].fit_transform(df)
kmeans_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
df_kmeans_5 = df.copy()
df_kmeans_5['cluster'] = kmeans_5.fit_predict(X_processed_5)

scatter2 = axes[1].scatter(
    df_kmeans_5['internalFacilitiesCount'], 
    df_kmeans_5['hospitals_10km'], 
    c=df_kmeans_5['cluster'], 
    cmap='viridis', 
    alpha=0.7,
    s=80
)
axes[1].set_title('K-Means (k=5) - Silhouette Score: 0.65')
axes[1].set_xlabel('Internal Facilities Count')
axes[1].set_ylabel('Hospitals within 10km')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
city_order = df_kmeans.groupby('city')['qualityTier'].count().sort_values(ascending=False).index

sns.countplot(
    data=df_kmeans, 
    x='city', 
    hue='qualityTier', 
    order=city_order,
    palette={'Premium': 'green', 'Standard': 'orange', 'Basic': 'red'}
)
plt.title('Distribution of Research Center Tiers across Cities')
plt.xlabel('City')
plt.ylabel('Number of Research Centers')
plt.legend(title='Tier')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = [
    'internalFacilitiesCount',
    'hospitals_10km',
    'pharmacies_10km',
    'facilityDiversity_10km',
    'facilityDensity_10km'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    sns.boxplot(
        data=df_kmeans, 
        x='qualityTier', 
        y=feature, 
        order=['Premium', 'Standard', 'Basic'],
        palette={'Premium': 'green', 'Standard': 'orange', 'Basic': 'red'},
        ax=axes[i]
    )
    axes[i].set_title(f'{feature} by Tier')
    axes[i].set_xlabel('Quality Tier')
    axes[i].set_ylabel(feature)

# Remove extra subplot
if len(features_to_plot) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

Most important

In [ ]:
tier_means = df_kmeans.groupby('qualityTier')[features_to_plot].mean()
tier_means = tier_means.reindex(['Premium', 'Standard', 'Basic'])

tier_means.plot(kind='bar', figsize=(12, 6))
plt.title('Average Feature Values by Quality Tier')
plt.xlabel('Quality Tier')
plt.ylabel('Average Value')
plt.legend(title='Features', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
n_runs = 10
random_seeds = [42, 123, 456, 789, 101112, 131415, 161718, 192021, 222324, 252627]

X_processed_stability = experiments[mvp_exp].fit_transform(df)

cluster_labels_stability = []
for seed in random_seeds:
    kmeans_temp = KMeans(n_clusters=3, random_state=seed, n_init=10)
    labels_temp = kmeans_temp.fit_predict(X_processed_stability)
    cluster_labels_stability.append(labels_temp)

labels_df_stability = pd.DataFrame(cluster_labels_stability).T
mode_labels_stability = labels_df_stability.mode(axis=1)[0]
stability_scores = (labels_df_stability == mode_labels_stability.values.reshape(-1, 1)).mean(axis=1)

print(f"Average cluster stability: {stability_scores.mean():.3f}")
print(f"Stability range: {stability_scores.min():.3f} to {stability_scores.max():.3f}")

plt.figure(figsize=(10, 4))
plt.hist(stability_scores, bins=20, alpha=0.7, edgecolor='black')
plt.axvline(x=stability_scores.mean(), color='red', linestyle='--', label=f'Mean: {stability_scores.mean():.3f}')
plt.xlabel('Stability (proportion of runs with same cluster)')
plt.ylabel('Number of Research Centers')
plt.title('Cluster Assignment Stability Across Multiple Runs')
plt.legend()
plt.show()

In [ ]:
features_for_pca = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km',
                    'facilityDiversity_10km', 'facilityDensity_10km']

X_pca_input = StandardScaler().fit_transform(df_kmeans[features_for_pca])
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_pca_input)

df_pca = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
df_pca['qualityTier'] = df_kmeans['qualityTier'].values

plt.figure(figsize=(9, 6))
palette = {'Premium': 'green', 'Standard': 'orange', 'Basic': 'red'}
for tier, grp in df_pca.groupby('qualityTier'):
    plt.scatter(grp['PC1'], grp['PC2'], label=tier, color=palette[tier], alpha=0.8, s=80, edgecolors='k', linewidths=0.4)

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('PCA 2D Projection of Clusters')
plt.legend(title='Quality Tier')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("PCA Explained Variance Ratio:", pca_2d.explained_variance_ratio_.round(3))
print("Cumulative:", pca_2d.explained_variance_ratio_.cumsum().round(3))

In [ ]:
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=features_for_pca,
    columns=['PC1', 'PC2']
)

fig, ax = plt.subplots(figsize=(8, 5))
loadings.plot(kind='bar', ax=ax, color=['steelblue', 'coral'], edgecolor='black')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('PCA Feature Loadings (contribution to each component)')
ax.set_ylabel('Loading')
ax.set_xlabel('Feature')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Component')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
features_radar = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km',
                  'facilityDiversity_10km', 'facilityDensity_10km']

tier_means_radar = df_kmeans.groupby('qualityTier')[features_radar].mean()
tier_means_norm = (tier_means_radar - tier_means_radar.min()) / (tier_means_radar.max() - tier_means_radar.min())

labels = features_radar
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors_radar = {'Premium': 'green', 'Standard': 'orange', 'Basic': 'red'}

for tier in ['Premium', 'Standard', 'Basic']:
    values = tier_means_norm.loc[tier].tolist()
    values += values[:1]
    ax.plot(angles, values, label=tier, color=colors_radar[tier], linewidth=2)
    ax.fill(angles, values, color=colors_radar[tier], alpha=0.1)

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Normalised Feature Profile by Quality Tier', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

In [ ]:
centroid_df = df_kmeans.groupby('qualityTier')[features_for_pca].mean().reindex(['Premium', 'Standard', 'Basic'])
centroid_norm = (centroid_df - centroid_df.min()) / (centroid_df.max() - centroid_df.min())

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(centroid_norm, annot=centroid_df.round(2), fmt='g',
            cmap='RdYlGn', linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalised value'})
ax.set_title('Cluster Centroid Heatmap (annotated with raw means, coloured by normalised value)')
ax.set_xlabel('Feature')
ax.set_ylabel('Quality Tier')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial.distance import cdist

features_outlier = ['internalFacilitiesCount', 'hospitals_10km', 'pharmacies_10km',
                    'facilityDiversity_10km', 'facilityDensity_10km']

X_scaled = StandardScaler().fit_transform(df_kmeans[features_outlier])
centroids = np.array([X_scaled[df_kmeans['qualityTier'] == tier].mean(axis=0)
                      for tier in ['Premium', 'Standard', 'Basic']])
tier_to_idx = {'Premium': 0, 'Standard': 1, 'Basic': 2}

df_kmeans['dist_to_centroid'] = [
    np.linalg.norm(X_scaled[i] - centroids[tier_to_idx[tier]])
    for i, tier in enumerate(df_kmeans['qualityTier'])
]

threshold = df_kmeans.groupby('qualityTier')['dist_to_centroid'].transform(
    lambda x: x.mean() + 1.5 * x.std()
)
df_kmeans['is_borderline'] = df_kmeans['dist_to_centroid'] > threshold

fig, ax = plt.subplots(figsize=(9, 6))
for tier, grp in df_kmeans.groupby('qualityTier'):
    core = grp[~grp['is_borderline']]
    border = grp[grp['is_borderline']]
    ax.scatter(core['internalFacilitiesCount'], core['hospitals_10km'],
               color=palette[tier], alpha=0.8, s=70, label=f'{tier} (core)')
    ax.scatter(border['internalFacilitiesCount'], border['hospitals_10km'],
               color=palette[tier], alpha=0.9, s=130, marker='*', edgecolors='black',
               linewidths=0.8, label=f'{tier} (borderline)')

ax.set_xlabel('Internal Facilities Count')
ax.set_ylabel('Hospitals within 10km')
ax.set_title('Core vs Borderline Centers per Tier\n(★ = centres furthest from their cluster centroid)')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBorderline centers ({df_kmeans['is_borderline'].sum()} total):")
df_kmeans[df_kmeans['is_borderline']][['researchCenterName', 'city', 'qualityTier', 'dist_to_centroid']].sort_values('dist_to_centroid', ascending=False)

In [ ]:
df_kmeans.groupby('qualityTier').agg(**{
    'Total Internal Facilities': ('internalFacilitiesCount', 'sum'),
    'Total Hospitals': ('hospitals_10km', 'sum'),
    'Total Pharmacies': ('pharmacies_10km', 'sum'),
    'Average Facility Diversity': ('facilityDiversity_10km', 'mean'),
    'Average Facility Density': ('facilityDensity_10km', 'mean')
})

# Save Model

In [ ]:
final_pipeline_k3 = Pipeline([
    ('preprocessor', experiments[mvp_exp]), # e.g., geohash_5_pca_2
    ('kmeans', KMeans(n_clusters=3, random_state=42, n_init=10))
])
final_pipeline_k3.fit(df)

# Save the model for the API
joblib.dump(final_pipeline_k3, 'models/final_kmeans_pipeline.pkl')

print("--- Comparison of 3-Cluster vs. 5-Cluster Models ---")
print(f"3-Cluster Silhouette Score: {master_results_df.loc[master_results_df['experiment'] == mvp_exp, 'silhouette'].values[0]}")
print(f"5-Cluster Silhouette Score: {master_results_df.sort_values(by='silhouette', ascending=False).reset_index(drop=True).loc[0, 'silhouette']}")


## API Testing Guide

curl http://127.0.0.1:8000/health

curl -X POST "http://127.0.0.1:8000/predict" -H "Content-Type: application/json" -d "{\"internalFacilitiesCount\": 9, \"hospitals_10km\": 3, \"pharmacies_10km\": 2, \"facilityDiversity_10km\": 0.82, \"facilityDensity_10km\": 0.45}"



# Summary and Recommendations

### The Story in Four Layers

#### Layer 1: The Problem

**Headline: 141% Improvement Over Baseline – Moving from Random Assignment to Actionable Segmentation**

I needed to solve a practical business problem: where should new research centers be developed to maximize success? Without a clear understanding of existing facility quality, investment decisions risk being placed in oversaturated areas or resource-poor locations. My goal was to develop a model that categorizes UK research centers into quality tiers—Premium, Standard, and Basic—based on internal infrastructure and access to external healthcare services. This enables stakeholders to identify optimal locations for new developments, either in under-served areas (where Basic centers lack resources) or in resource-rich zones where competition for a particular tier is lower.

---

#### Layer 2: The Process – My Methodology

**From Raw Data to Actionable Insight**

I analyzed 50 synthetic research centers across five UK cities. Each center had attributes measuring internal facilities and nearby healthcare services.

**Step 1: Exploratory Data Analysis**
- I visualized distributions, correlations, and geographic patterns to understand the data structure.
- **Key finding:** `internalFacilitiesCount` correlated with `facilityDiversity_10km` at 0.90 and with `external_score` at 0.93. This confirmed that centers with more internal facilities also have better access to diverse external resources—a strong signal of quality.

**Step 2: Feature Selection**
- I selected five core features: `internalFacilitiesCount`, `hospitals_10km`, `pharmacies_10km`, `facilityDiversity_10km`, and `facilityDensity_10km`.
- **Why these features?** They represent two pillars of research center success: internal capability (facilities count) and external support (healthcare access). The correlation heatmap confirmed these features collectively explain most of the variance.

**Step 3: Geographic Encoding**
- Raw latitude/longitude performed poorly, confirming location matters non-linearly. I tested multiple transformations:
  - **Geohashing** (precision 2–9) to group centers by geographic proximity
  - **PCA** to reduce dimensionality while preserving signal
  - **Cartesian coordinates** for alternative spatial encoding
- The winning combination: **geohashing at precision 5 followed by PCA with 2 components**. This captured regional differences in healthcare infrastructure without overfitting.

**Step 4: Clustering**
- I applied **K-Means** with k=3 (as required) and evaluated using silhouette scores.
- **Baseline comparison:** A Gaussian Mixture Model (GMM) with default preprocessing scored **0.2525**.
- My best model achieved a **silhouette score of 0.6097**—a **141% improvement** over baseline. This means the clusters are far more coherent and reliable for business decisions.

**Step 5: Discovery – The 5-Cluster Structure**
- When I tested k=5, the silhouette score increased to **0.65**. The data naturally separates into **five distinct tiers**. This suggests a more nuanced classification could be valuable in practice, revealing hidden opportunities: areas with high-quality centers that competitors may overlook, or lower-tier clusters where strategic investment could create outsized returns.

**Step 6: Validation**
- I compared my clustering against random assignments to ensure results were not spurious.
- Stability checks across multiple random seeds confirmed cluster assignments were consistent (>90% stability on average).

---

#### Layer 3: The Results – With Context

**Headline: Basic Centers Have Zero Healthcare Access – A Critical Business Finding**

| Metric | Baseline (GMM) | My Model (K-Means, geohash5+PCA2, k=3) | Improvement |
|--------|----------------|------------------------------------------|-------------|
| Silhouette Score | 0.2525 | 0.6097 | **+141%** |

This improvement translates directly to real-world consequences:

- **Basic centers** (lowest tier) had **zero hospitals or pharmacies within 10km**. This aligns with the business need: these facilities are truly isolated and likely struggle to attract talent or partnerships.
- **Premium centers** consistently had the highest internal facilities and greatest access to hospitals and pharmacies.
- **Standard centers** formed a clear middle group, offering opportunities for development where resources exist but competition is moderate.

**Geographic Patterns**
High-quality centers were not uniformly distributed. Some cities (e.g., City 3) hosted a mix of Premium and Standard centers, while others had only Basic ones. This tells me location *does* matter—but only when encoded appropriately. My geohashing approach captured these regional differences effectively.

**Diversity vs. Density**
Facility diversity (variety of nearby services) played a stronger role than raw density in separating Premium from Standard centers. This suggests a well-rounded environment (labs, hospitals, pharmacies) is more valuable than sheer quantity.

**Feature Importance**
- `internalFacilitiesCount` had the strongest influence on clustering
- `hospitals_10km` and `pharmacies_10km` were critical for separating Basic from other tiers
- `facilityDiversity_10km` was more discriminative than `facilityDensity_10km`

---

#### Layer 4: The Reflection – What I Would Improve

**Headline: From 3 Tiers to 5 – And Beyond**

**What Worked Well**
- Geohashing combined with PCA proved powerful, capturing geographic context effectively.
- The feature set (internal facilities + external services) captured the essence of center quality.
- Silhouette analysis gave me confidence that the clusters are meaningful.
- I Briefly did some outlier detection in order to identify borderline tiered research centers, not many were borderline which is good.

**What I Would Improve with More Time**

1. **Real geographic data**: Instead of synthetic coordinates, I would use a geocoding library to obtain actual town/city names and distances to real hospitals. This would make the model production‑ready and more interpretable.

2. **Granular tiers**: The 5‑cluster model outperforms the 3‑tier version. I would recommend using 5 tiers in practice to capture subtle differences, allowing stakeholders to identify niche opportunities (e.g., "upper‑standard" centers that could be upgraded to Premium with targeted investment).

3. **Continuous retraining**: The API endpoint could be extended to accept new data and retrain on a schedule (e.g., weekly), using a versioned model store and a background task to update the pipeline.

4. **External data**: I would add population density, proximity to universities, and transportation links to enrich features.

5. **Geocaching library integration**: I would use an actual geocaching library to get real town/city names for better interpretability.

**Why Clustering Is the Right Approach**
Clustering is ideal because I had no labeled data. It uncovers natural groupings that reflect real‑world quality differences, without bias from predefined labels. It also adapts as new centers are added.

---

## Q&A – Addressing the Key Questions

| Question | Answer |
|----------|--------|
| **Why were these features selected?** | They cover the two pillars of center quality: internal capacity (facilities count) and external support (hospitals, pharmacies, diversity, density). Correlation analysis confirmed they are highly inter‑correlated and together explain most of the variance. |
| **Which features have the highest correlation with overall facility diversity or quality?** | `internalFacilitiesCount` correlates with `facilityDiversity_10km` at 0.90 and with `external_score` at 0.93—making it the strongest single indicator. |
| **Which cluster has the highest internal facility counts and external healthcare access?** | The **Premium cluster**. Its mean internal facilities count is >10, with hospitals and pharmacies also highest. |
| **Are high-quality centers concentrated in specific cities?** | Yes. City 3 had a mix of Premium and Standard centers, while City 4 had only Basic. This suggests regional disparities that the model captures well. |
| **Does diversity or density play a stronger role?** | Diversity (`facilityDiversity_10km`) was more discriminative than density. A varied ecosystem (e.g., both hospitals and pharmacies) appears more important than raw number of facilities. |
| **Which features had the greatest influence on clustering?** | `internalFacilitiesCount` and `hospitals_10km` contributed most to separating the clusters, as shown by cluster means and PCA loadings. |
| **What patterns were visible in the data during EDA?** | Scatter plots showed a clear positive relationship between internal facilities and external services. Boxplots revealed that cities varied widely in facility counts, hinting at geographic effects. |
| **Why is clustering a good approach for this problem?** | No ground‑truth labels exist; clustering lets me discover natural tiers. It's unsupervised, avoids label bias, and can be reapplied as new centers appear. |
| **How would you improve this model if real data were available?** | Add more detailed location data (actual town/city names, road distances), incorporate population density and research institution proximity, and use a 5‑cluster model for finer granularity. |
| **How can the endpoint be extended for continuous retraining?** | Add a `/retrain` endpoint that accepts new data, retrains the pipeline in the background, saves the new model with a version number, and switches the active model only after validation. |
| **Bonus: How to commercialise and scale the solution?** | Package the API in a Docker container, deploy on cloud (AWS Lambda + API Gateway), add authentication, and offer a subscription‑based service. Provide a dashboard (e.g., Streamlit) for non‑technical users to explore tier maps. Partner with real estate or investment firms to integrate into their site‑selection tools. |

---

## Conclusion

My model transforms raw facility data into actionable business intelligence. The **141% silhouette improvement** over baseline means I can now confidently identify Basic centers that lack critical healthcare infrastructure—information that can directly influence where to build new facilities. By uncovering a 5‑tier structure, I also revealed opportunities to discover niche areas where high‑quality centers cluster, giving stakeholders a competitive edge.

If given more time, I would enrich the data with real geography and transition to a 5‑tier system, delivering even finer insights. The API is ready to scale, and with containerization and continuous retraining, this solution can become a core decision‑support tool for research center development.m